# 🩺 Mini-ChatGPT na nazwach chorób
### Sympozjum „Nowoczesne technologie w pediatrii" · Kamil Jędryczek

Ten notebook buduje **mały model językowy** — ten sam mechanizm co ChatGPT, tylko ~100 miliardów razy mniejszy. Zamiast następnego **słowa**, przewidujemy następną **literę** w łacińskiej nazwie choroby.

**Jak uruchomić:** `Środowisko wykonawcze` → `Uruchom wszystko` (Ctrl+F9). Trening na CPU ok. **15 s** (12 epok).

*Aplikację dla sali uruchamia prelegent: `notebook/uruchom_demo.py`.*

In [1]:
# ── KONFIGURACJA ──────────────────────────────────────────────
import os
import random

os.environ.setdefault("PYTHONHASHSEED", "42")
os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", "1")
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "1")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

# Który zbiór? "choroby" | "leki" | "dinozaury"
ZBIOR = "choroby"

REPO = "https://raw.githubusercontent.com/NXTRSS/sympozjum-pediatria/main/data"
DANE = {
    "choroby": f"{REPO}/choroby.csv",
    "leki": f"{REPO}/leki.csv",
    "dinozaury": "https://raw.githubusercontent.com/junosuarez/dinosaurs/master/dinosaurs.csv",
}

# Nastawy sympozjum (eksperymenty/AUTOOPTYMALIZACJA.md)
LICZBA_EPOK = 12
TEMPO_UCZENIA = 0.003
TEMPERATURA = 0.4
ZIARNO = 42

import numpy as np
import pandas as pd

random.seed(ZIARNO)
np.random.seed(ZIARNO)

In [2]:
# ── DANE ──────────────────────────────────────────────────────
from pathlib import Path

zrodlo = DANE[ZBIOR]
plik = os.path.basename(zrodlo)
for kandydat in (
    Path(plik),
    Path("data") / plik,
    Path("..") / "data" / plik,
    Path.cwd() / "data" / plik,
    Path.cwd().parent / "data" / plik,
):
    if kandydat.exists():
        zrodlo = str(kandydat)
        break

df = pd.read_csv(zrodlo, header=None, names=["nazwa"], sep="\t")
nazwy = (
    df["nazwa"].dropna().astype(str).str.strip().str.lower().drop_duplicates().tolist()
)
rng = np.random.default_rng(ZIARNO)
nazwy = [nazwy[i] for i in rng.permutation(len(nazwy))]

print(f"Wczytano {len(nazwy)} nazw z: {zrodlo}\nPrzykłady:")
for n in nazwy[:8]:
    print("  •", n)

Wczytano 1292 nazw z: ../data/choroby.csv
Przykłady:
  • ataxia teleangiectasia
  • cephalhematoma
  • dermatitis atopica
  • claudicatio intermittens
  • adenocarcinoma
  • trachoma
  • febris haemorrhagica cum nephropathia
  • carcinoma anaplasticum glandulae thyroideae


## Tokeny specjalne

Każdą nazwę opakowujemy w dwa znaki:
- `%` — **START** („zacznij generować")
- `!` — **STOP** („koniec nazwy")

`pneumonia` → `%pneumonia!`. Bez `%` model nie wie, od czego zacząć; bez `!` pisałby w nieskończoność.

In [3]:
# ── PRZYGOTOWANIE DANYCH ──────────────────────────────────────
sekwencje = ["%" + n + "!" for n in nazwy]

# Słownik: znak → indeks (0 = padding w Embedding)
znaki = sorted(set("".join(sekwencje)))
znak2id = {z: i + 1 for i, z in enumerate(znaki)}
id2znak = {i: z for z, i in znak2id.items()}
WIELKOSC_SLOWNIKA = len(znaki) + 1
print(f"Słownik ma {WIELKOSC_SLOWNIKA} tokenów: {''.join(znaki)}")
print("Przykłady znak → indeks:", {z: znak2id[z] for z in ("%", "!", "a") if z in znak2id})

# Kodowanie: litery zamieniamy na liczby (wejście Embedding)
przyklad = nazwy[0]
tekst = "%" + przyklad + "!"
indeksy = [znak2id[z] for z in tekst]
print(f"\nKodowanie {przyklad!r}: {tekst} → indeksy {indeksy}")

# X = prefiks, Y = następny znak (jak na slajdzie X/Y)
maks_dl = max(len(s) for s in sekwencje)
X = np.zeros((len(sekwencje), maks_dl - 1), dtype="int32")
Y = np.zeros((len(sekwencje), maks_dl - 1), dtype="int32")
for i, s in enumerate(sekwencje):
    ids = [znak2id[z] for z in s]
    X[i, : len(ids) - 1] = ids[:-1]
    Y[i, : len(ids) - 1] = ids[1:]
print(f"\nMacierz treningowa: {X.shape} (nazwy × pozycje)")

# Jedna nazwa — trzy pary X→Y (jak %otitis externa na slajdzie)
for j, s in enumerate(sekwencje):
    if " " in s:
        idx = j
        break
else:
    idx = 0
for poz in range(3):
    prefiks = "".join(id2znak[k] for k in X[idx, : poz + 1] if k)
    nast = id2znak[Y[idx, poz]]
    print(f"  X: {prefiks!r}  →  Y: {nast!r}")

Słownik ma 32 tokenów:  !%'-abcdefghijklmnopqrstuvwxyz
Przykłady znak → indeks: {'%': 3, '!': 2, 'a': 6}

Kodowanie 'ataxia teleangiectasia': %ataxia teleangiectasia! → indeksy [3, 6, 25, 6, 29, 14, 6, 1, 25, 10, 17, 10, 6, 19, 12, 14, 10, 8, 25, 6, 24, 14, 6, 2]

Macierz treningowa: (1292, 60) (nazwy × pozycje)
  X: '%'  →  Y: 'a'
  X: '%a'  →  Y: 't'
  X: '%at'  →  Y: 'a'


## Model

1. **Embedding(64)** — każda litera to wektor liczb  
2. **GRU(128)** z dropout 0.2 — pamięć kontekstu wzdłuż nazwy  
3. **Dense + softmax** — rozkład **następnej** litery  

ChatGPT ma Transformera i miliardy parametrów — tu jest ten sam schemat *kontekst → P(następny token)*.

In [4]:
# ── DEFINICJA MODELU ─────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras

try:
    tf.config.set_visible_devices([], "GPU")
except Exception:
    pass
tf.random.set_seed(ZIARNO)
keras.utils.set_random_seed(ZIARNO)
tf.config.experimental.enable_op_determinism()

model = keras.Sequential([
    keras.layers.Embedding(
        WIELKOSC_SLOWNIKA, 64, mask_zero=True,
        embeddings_initializer=keras.initializers.GlorotUniform(seed=ZIARNO),
    ),
    keras.layers.GRU(
        128, return_sequences=True, dropout=0.2, seed=ZIARNO,
        kernel_initializer=keras.initializers.GlorotUniform(seed=ZIARNO),
        recurrent_initializer=keras.initializers.Orthogonal(seed=ZIARNO + 1),
        bias_initializer=keras.initializers.Zeros(),
    ),
    keras.layers.Dense(
        WIELKOSC_SLOWNIKA, activation="softmax",
        kernel_initializer=keras.initializers.GlorotUniform(seed=ZIARNO + 2),
        bias_initializer=keras.initializers.Zeros(),
    ),
])
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=TEMPO_UCZENIA),
)
model.summary()

E0000 00:00:1795760733.217813   33163 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1795760733.222137   33163 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1795760733.235836   33163 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1795760733.235854   33163 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1795760733.235856   33163 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1795760733.235857   33163 computation_placer.cc:177] computation placer already registered. Please check linka

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [5]:
# ── TRENING ───────────────────────────────────────────────────
historia = model.fit(X, Y, epochs=LICZBA_EPOK, shuffle=True, verbose=2)

Epoch 1/12


41/41 - 3s - 64ms/step - loss: 2.7580


Epoch 2/12


41/41 - 1s - 23ms/step - loss: 2.2888


Epoch 3/12


41/41 - 1s - 23ms/step - loss: 2.1677


... (14 outputs left out)

Epoch 11/12


41/41 - 1s - 23ms/step - loss: 1.5267


Epoch 12/12


41/41 - 1s - 23ms/step - loss: 1.4675


In [6]:
# ── KRZYWA UCZENIA ────────────────────────────────────────────
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "notebook"))
sys.path.insert(0, str(Path.cwd()))
from lm_wizualizacja import wykres_krzywy_bledu

wykres_krzywy_bledu(historia)

/workspace/notebook/lm_wizualizacja.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Generowanie — serce autoregresji

1. Kontekst od `%`  
2. Model zwraca **rozkład** następnych liter — pokazujemy histogram (jak slajd o10)  
3. **Losujemy** z rozkładu (`TEMPERATURA=0.4` = ostrożniej)  
4. Dopisujemy literę — powtarzamy, aż wypadnie `!`  

Model **nigdy nie mówi „nie wiem"** — zawsze coś wybiera.

In [7]:
# ── GENEROWANIE ───────────────────────────────────────────────
from lm_wizualizacja import histogramy_krokow, pokaz_kroki, wykres_topk_prefiks

ZNANE = set(nazwy)

# Podgląd jednego prefiksu (histogram jak na slajdzie inferencji)
prefiks_demo = "%pneumon"
ids_demo = np.array([[znak2id[z] for z in prefiks_demo]])
probs_demo = model(ids_demo, training=False).numpy()[0, -1].astype("float64")
probs_demo[0] = 0
probs_demo /= probs_demo.sum()
print(f"P(następna litera | {prefiks_demo!r}) — top wartości:")
top = np.argsort(probs_demo)[-8:][::-1]
for k in top:
    print(f"  {id2znak[k]!r}: {probs_demo[k]:.3f}")
wykres_topk_prefiks(prefiks_demo, probs_demo, id2znak)


def generuj(temperatura=TEMPERATURA, maks_znakow=40):
    """Zwraca (nazwa, kroki). Pętla: softmax → temperatura → losowanie."""
    kontekst, kroki = "%", []
    for _ in range(maks_znakow):
        ids = np.array([[znak2id[z] for z in kontekst]])
        rozklad = model(ids, training=False).numpy()[0, -1].astype("float64")
        rozklad = np.log(rozklad + 1e-9) / temperatura
        rozklad = np.exp(rozklad)
        rozklad[0] = 0
        rozklad /= rozklad.sum()
        wybor = int(np.random.choice(len(rozklad), p=rozklad))
        kroki.append({"kontekst": kontekst, "rozklad": rozklad, "wybor": wybor})
        if id2znak[wybor] == "!":
            break
        kontekst += id2znak[wybor]
    return kontekst[1:], kroki


pierwsza_nazwa, kroki = generuj()
print("Ślad generowania — krok po kroku:\n")
pokaz_kroki(kroki, id2znak)
print("\nWynik:", pierwsza_nazwa)
print(
    "⚠️ już była w zbiorze" if pierwsza_nazwa in ZNANE
    else "✨ nowa względem zbioru treningowego"
)
histogramy_krokow(kroki, id2znak, WIELKOSC_SLOWNIKA)

P(następna litera | '%pneumon') — top wartości:
  'e': 0.324
  'i': 0.261
  'a': 0.210
  'u': 0.056
  'o': 0.055
  'y': 0.032
  't': 0.018
  '!': 0.013


/workspace/notebook/lm_wizualizacja.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Ślad generowania — krok po kroku:

  kontekst: %                            → wylosowano 'e'   (p=3%)   top5: 's':24%  'a':20%  'p':15%  'c':12%  'm':7%
  kontekst: %e                           → wylosowano 'r'   (p=4%)   top5: 'n':83%  'p':10%  'r':4%  'x':2%  'm':0%
  kontekst: %er                          → wylosowano 'y'   (p=97%)   top5: 'y':97%  'a':1%  't':1%  'o':0%  'p':0%
  kontekst: %ery                         → wylosowano 't'   (p=99%)   top5: 't':99%  'p':0%  's':0%  'e':0%  'n':0%
  kontekst: %eryt                        → wylosowano 'h'   (p=100%)   top5: 'h':100%  'o':0%  'i':0%  'e':0%  'r':0%
  kontekst: %eryth                       → wylosowano 'e'   (p=87%)   top5: 'e':87%  'r':12%  'o':1%  'i':0%  'a':0%
  kontekst: %erythe                      → wylosowano 'm'   (p=99%)   top5: 'm':99%  'r':1%  's':0%  'a':0%  'n':0%
  kontekst: %erythem                     → wylosowano 'a'   (p=100%)   top5: 'a':100%  'i':0%  'o':0%  'e':0%  'm':0%
  kontekst: %erythema         

/workspace/notebook/lm_wizualizacja.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# ── 10 NOWYCH NAZW ────────────────────────────────────────────
lista_nazw = []
nowe = 0
for _ in range(10):
    nazwa, _ = generuj(temperatura=TEMPERATURA)
    lista_nazw.append(nazwa)
    if nazwa in ZNANE:
        print(f" • {nazwa:<40} ⚠️  istnieje w zbiorze treningowym")
    else:
        print(f" • {nazwa:<40} ✨ nowa")
        nowe += 1
print(f"\n{nowe}/10 nazw nie występuje w zbiorze treningowym.")

 • carcinoma cortica                        ✨ nowa


 • analitis atuta                           ✨ nowa


 • steatophia contica                       ✨ nowa


 • syndroma conteri                         ✨ nowa


 • syndroma contonica                       ✨ nowa


 • carcinoma praematocalia teriosa          ✨ nowa


 • morbus hiparticularis                    ✨ nowa


 • aphalia prograns angiosa                 ✨ nowa


 • carcinoma polita                         ✨ nowa


 • anaemia praventriculorum                 ✨ nowa

10/10 nazw nie występuje w zbiorze treningowym.
